In [2]:
import pandas as pd
import numpy as np
from scipy.stats import mode

# Fonction pour calculer le κ de Krippendorff
def krippendorff_alpha(data):
    n = len(data)
    if n == 0:
        return np.nan
    unique_values = np.unique(data)
    n_values = len(unique_values)
    if n_values == 1:
        return 1.0
    observed = np.zeros((n_values, n_values))
    for i, j in zip(data[:-1], data[1:]):
        observed[i-1, j-1] += 1
    observed = observed + observed.T - np.diag(np.diag(observed))
    expected = np.outer(observed.sum(axis=1), observed.sum(axis=0)) / (n - 1)
    chance = np.sum(np.outer(observed.sum(axis=1), observed.sum(axis=0)) / ((n - 1) ** 2))
    alpha = 1 - np.sum(observed * np.log(observed / expected)) / np.sum(observed * np.log(chance))
    return alpha

In [3]:
df = pd.read_parquet("data/reactions.parquet")
df.head()

,id,timestamp,model_a_name,model_b_name,refers_to_model,msg_index,opening_msg,conversation_a,conversation_b,model_pos,...,creative,complete,clear_formatting,incorrect,superficial,instructions_not_followed,model_pair_name,msg_rank,question_id,system_prompt
0,202099,2025-04-23 19:32:46.687338,gemma-3-12b,command-a,command-a,1,Quelle est la stratégie commerciale du champag...,[{'content': 'Quelle est la stratégie commerci...,[{'content': 'Quelle est la stratégie commerci...,b,...,False,True,False,False,False,False,"[command-a, gemma-3-12b]",0,1a62544c50474085b25240991e9fc620-6e6ffca415f44...,NaN
1,176467,2025-03-25 13:41:21.988182,llama-3.1-8b,claude-3-5-sonnet-v2,llama-3.1-8b,1,Qui est président en France ?,"[{'content': 'Qui est président en France ?', ...","[{'content': 'Qui est président en France ?', ...",a,...,False,True,False,False,False,False,"[claude-3-5-sonnet-v2, llama-3.1-8b]",0,ba089364a6e64d14b80eae6f7dd1885f-cda152aa7fe04...,NaN
2,231166,2025-06-03 15:38:11.580715,gpt-4.1-nano,llama-4-scout,llama-4-scout,1,tests logiciel,"[{'content': 'tests logiciel', 'metadata': Non...","[{'content': 'tests logiciel', 'metadata': Non...",b,...,False,True,False,False,False,False,"[gpt-4.1-nano, llama-4-scout]",0,e46e59300e8545038bfc22dfffb0b746-b006443eb1564...,NaN
3,264589,2025-08-20 13:52:02.676745,llama-3.1-405b,mistral-large-2411,llama-3.1-405b,13,Vols au meilleur prix au départ de Marseille p...,[{'content': 'Vols au meilleur prix au départ ...,[{'content': 'Vols au meilleur prix au départ ...,a,...,False,False,False,False,False,False,"[llama-3.1-405b, mistral-large-2411]",6,1f181e340b664f88b1aacc6770d0eeff-3f295419957c4...,NaN
4,203152,2025-04-25 20:40:31.489534,o4-mini,claude-3-7-sonnet,o4-mini,1,Comporte toi comme un expert en rédaction de p...,[{'content': 'Comporte toi comme un expert en ...,[{'content': 'Comporte toi comme un expert en ...,a,...,False,False,False,False,True,False,"[claude-3-7-sonnet, o4-mini]",0,5a646fd23c7c480dbbb9dac5f50b8c6d-5d343345ef294...,NaN


In [5]:
counts = df.groupby(
      ['conversation_pair_id','msg_index'])['creative'].nunique()
print(counts.describe())
eligible = counts[counts >= 2].index
sub = df.set_index(
    ['conversation_pair_id','msg_index']).loc[eligible].reset_index()
sub[['conversation_pair_id','msg_index', "visitor_id", "creative"]].head()

count    59965.000000
mean         1.043992
std          0.232016
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          2.000000
Name: creative, dtype: float64


,conversation_pair_id,msg_index,visitor_id,creative
0,00034204d6994d8e99070b75b4fb3f67-2e5d7407743f4...,1,fc56866e083476414e83d6f9e3c7f278,False
1,00034204d6994d8e99070b75b4fb3f67-2e5d7407743f4...,1,fc56866e083476414e83d6f9e3c7f278,True
2,00184390104b46acbb5bd65c161600f8-b587ed8a736a4...,1,21b266c448dd401eb517a38609da241c,True
3,00184390104b46acbb5bd65c161600f8-b587ed8a736a4...,1,21b266c448dd401eb517a38609da241c,False
4,001b5be61db148af9c2d5f165ba01810-50a0ca5693684...,1,693405628a454c1808b52245d94c88ba,False


In [ ]:
# Filtrer les messages évalués par au moins 3 utilisateurs distincts
message_counts = df.groupby(["response_content"])['visitor_id'].nunique()
valid_messages = message_counts[message_counts >= 3].index
filtered_df = df[df[["response_content"]].isin(valid_messages)]
print(filtered_df)

       id timestamp model_a_name model_b_name refers_to_model  msg_index  \
0     NaN       NaT          NaN          NaN             NaN        NaN   
1     NaN       NaT          NaN          NaN             NaN        NaN   
2     NaN       NaT          NaN          NaN             NaN        NaN   
3     NaN       NaT          NaN          NaN             NaN        NaN   
4     NaN       NaT          NaN          NaN             NaN        NaN   
...    ..       ...          ...          ...             ...        ...   
91828 NaN       NaT          NaN          NaN             NaN        NaN   
91829 NaN       NaT          NaN          NaN             NaN        NaN   
91830 NaN       NaT          NaN          NaN             NaN        NaN   
91831 NaN       NaT          NaN          NaN             NaN        NaN   
91832 NaN       NaT          NaN          NaN             NaN        NaN   

      opening_msg conversation_a conversation_b model_pos  ...  creative  \
0          

In [ ]:

# Calculer le κ de Krippendorff pour chaque catégorie
results = {}
for category in ['creative', 'useful', 'incorrect']:
    grouped = filtered_df.groupby(["response_content"])[category]
    alphas = []
    for _, group in grouped:
        alphas.append(krippendorff_alpha(group.values))
    results[category] = np.mean(alphas)

results